# Demand Rising Notebook

## Import Libraries and Dataset

In [1]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


import os
from dotenv import load_dotenv
from pathlib import Path

from sqlalchemy import create_engine, text

import warnings
warnings.filterwarnings('ignore')

In [2]:
env_path = Path.cwd()/"github_repo_cloned"/"festival_restocking_problem"/".env"
load_dotenv(env_path)


host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
port = os.getenv("DB_PORT")
password = os.getenv("DB_PASSword")
db = os.getenv("DB_NAME")

In [3]:
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}")

def sql(query, engine = engine):
    with engine.connect() as conn:
        df = pd.read_sql(text(query),conn)
    return df

In [4]:
data = sql('select * from festival_weekly_base')

In [5]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1.0,Diwali,MH,2020-11-14,25,1.03
1,2020-05-24,1.0,Diwali,MH,2020-11-14,24,1.03
2,2020-05-31,1.0,Diwali,MH,2020-11-14,23,1.03
3,2020-06-07,1.0,Diwali,MH,2020-11-14,22,1.03
4,2020-06-14,1.0,Diwali,MH,2020-11-14,21,1.03


## Initial Data Checking

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44880 entries, 0 to 44879
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   week             44880 non-null  object 
 1   interest         44880 non-null  float64
 2   festival         44880 non-null  object 
 3   state            44880 non-null  object 
 4   festival_date    44880 non-null  object 
 5   wk_before        44880 non-null  int64  
 6   normal_interest  44880 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 2.4+ MB


In [7]:
data[['week','festival_date']] = data[['week','festival_date']].apply(lambda x: pd.to_datetime(x))

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44880 entries, 0 to 44879
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   week             44880 non-null  datetime64[ns]
 1   interest         44880 non-null  float64       
 2   festival         44880 non-null  object        
 3   state            44880 non-null  object        
 4   festival_date    44880 non-null  datetime64[ns]
 5   wk_before        44880 non-null  int64         
 6   normal_interest  44880 non-null  float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(2)
memory usage: 2.4+ MB


In [9]:
data.isnull().sum()

week               0
interest           0
festival           0
state              0
festival_date      0
wk_before          0
normal_interest    0
dtype: int64

In [10]:
data.duplicated().sum()

np.int64(0)

In [11]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1.0,Diwali,MH,2020-11-14,25,1.03
1,2020-05-24,1.0,Diwali,MH,2020-11-14,24,1.03
2,2020-05-31,1.0,Diwali,MH,2020-11-14,23,1.03
3,2020-06-07,1.0,Diwali,MH,2020-11-14,22,1.03
4,2020-06-14,1.0,Diwali,MH,2020-11-14,21,1.03


In [12]:
data[['interest','wk_before']] = data[['interest','wk_before']].apply(lambda x: x.astype(int))

In [13]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1,Diwali,MH,2020-11-14,25,1.03
1,2020-05-24,1,Diwali,MH,2020-11-14,24,1.03
2,2020-05-31,1,Diwali,MH,2020-11-14,23,1.03
3,2020-06-07,1,Diwali,MH,2020-11-14,22,1.03
4,2020-06-14,1,Diwali,MH,2020-11-14,21,1.03


In [14]:
data['state'].unique()

array(['MH', 'WB', 'KL', 'TN', 'UP', 'GJ', 'KA', 'PB', 'DL', 'RJ', 'AS',
       'BR', 'TG', 'AP', 'OD'], dtype=object)

In [15]:
state_names = {
    'MH': 'Maharashtra', 'WB': 'West Bengal', 'KL': 'Kerala',
    'TN': 'Tamil Nadu', 'UP': 'Uttar Pradesh', 'GJ': 'Gujarat',
    'KA': 'Karnataka', 'PB': 'Punjab', 'DL': 'Delhi',
    'RJ': 'Rajasthan', 'AS': 'Assam', 'BR': 'Bihar',
    'TG': 'Telangana', 'AP': 'Andhra Pradesh', 'OD': 'Odisha'
}

data['state'] = data['state'].map(state_names)

In [16]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1,Diwali,Maharashtra,2020-11-14,25,1.03
1,2020-05-24,1,Diwali,Maharashtra,2020-11-14,24,1.03
2,2020-05-31,1,Diwali,Maharashtra,2020-11-14,23,1.03
3,2020-06-07,1,Diwali,Maharashtra,2020-11-14,22,1.03
4,2020-06-14,1,Diwali,Maharashtra,2020-11-14,21,1.03


In [17]:
data = data[data['wk_before']>=0]

In [18]:
data.shape

(22725, 7)

In [23]:
data[data['wk_before'] == 0]

,week,interest,festival,state,festival_date,wk_before,normal_interest
25,2020-11-08,94,Diwali,Maharashtra,2020-11-14,0,1.03
77,2021-10-31,100,Diwali,Maharashtra,2021-11-04,0,1.06
129,2022-10-23,98,Diwali,Maharashtra,2022-10-24,0,1.11
182,2023-11-12,78,Diwali,Maharashtra,2023-11-12,0,1.11
233,2024-10-27,100,Diwali,Maharashtra,2024-10-31,0,1.09
...,...,...,...,...,...,...,...
44645,2021-04-11,49,Bihu (Rongali),Odisha,2021-04-14,0,8.34
44697,2022-04-10,53,Bihu (Rongali),Odisha,2022-04-14,0,14.26
44749,2023-04-09,54,Bihu (Rongali),Odisha,2023-04-14,0,14.42
44802,2024-04-14,28,Bihu (Rongali),Odisha,2024-04-14,0,12.79


In [27]:
data[(data['festival']=='Bihu (Rongali)') & (data['state']=='Odisha') & (data['week'].dt.year==2025)]

,week,interest,festival,state,festival_date,wk_before,normal_interest
44839,2025-01-05,27,Bihu (Rongali),Odisha,2025-04-14,14,10.63
44840,2025-01-12,58,Bihu (Rongali),Odisha,2025-04-14,13,10.63
44841,2025-01-19,16,Bihu (Rongali),Odisha,2025-04-14,12,10.63
44842,2025-01-26,12,Bihu (Rongali),Odisha,2025-04-14,11,10.63
44843,2025-02-02,0,Bihu (Rongali),Odisha,2025-04-14,10,10.63
44844,2025-02-09,0,Bihu (Rongali),Odisha,2025-04-14,9,10.63
44845,2025-02-16,0,Bihu (Rongali),Odisha,2025-04-14,8,10.63
44846,2025-02-23,22,Bihu (Rongali),Odisha,2025-04-14,7,10.63
44847,2025-03-02,0,Bihu (Rongali),Odisha,2025-04-14,6,10.63
44848,2025-03-09,19,Bihu (Rongali),Odisha,2025-04-14,5,10.63


## Data Modification

## Analysis